# DSA1 – Final cross-task comparison (all members)
Reads Task 2's result table (attach Aditya's notebook output) and answers the research question:
does a better score or a shift correction reduce the gap more? Target 10%, Qwen3-4B.

In [ ]:
# --- Setup (implementation guide, section 2). Run once per session. ------------------------------
import os, sys, subprocess, random, json
from pathlib import Path

REPO_URL = "https://github.com/adiarora28/dsa1-conformal-abstention.git"   # <- set once for the team
ON_KAGGLE = Path("/kaggle").exists()
REPO_DIR = Path("/kaggle/working/dsa1-conformal-abstention") if ON_KAGGLE else Path.cwd().resolve().parent
if ON_KAGGLE and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:  # upgrade only what Kaggle's image may lack; never reinstall torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.56",
                    "accelerate", "bitsandbytes", "sentence-transformers", "xgboost"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except ImportError:
    print("PyTorch not installed (this notebook can run on CPU)")

from dsa1 import config as C
from dsa1.cache_io import link_cache_roots
CACHE_ROOT = link_cache_roots("/kaggle/input", "/kaggle/working/cache_merged") if ON_KAGGLE else Path("cache")
print("CACHE_ROOT:", CACHE_ROOT)
if ON_KAGGLE:  # bring outputs of earlier notebooks (attached as inputs) into this working copy
    import shutil
    Path("results").mkdir(exist_ok=True)
    for p in Path("/kaggle/input").rglob("semantic_entropy"):
        if p.is_dir():
            shutil.copytree(p, "results/semantic_entropy", dirs_exist_ok=True)
    for name in ["hand_check.csv", "failure_cases.csv"]:
        hits = sorted(Path("/kaggle/input").rglob(name))
        if hits and not (Path("results") / name).exists():
            shutil.copy(hits[0], Path("results") / name)
            print("copied", hits[0])
for p in sorted(Path(CACHE_ROOT).glob("*/*")):
    print("  ", p.relative_to(CACHE_ROOT))


In [ ]:
hits = list(Path("/kaggle/input").rglob("task2_results.csv")) if ON_KAGGLE else []
path = hits[0] if hits else Path("results/metrics/task2_results.csv")
res = pd.read_csv(path)
from dsa1.metrics import mean_std
for test_set in ["mix_100", "mmlu_all"]:
    d = res[(res["model"] == "qwen3-4b") & (res["target"] == 0.10) & (res["test_set"] == test_set)]
    print(test_set, "– gap (realised − target) | coverage")
    display(d.pivot_table(index="score", columns="method", values="gap", aggfunc=mean_std))
    display(d.pivot_table(index="score", columns="method", values="coverage", aggfunc=mean_std))

Read the table: down the `uniform` column = effect of a better score alone; along the `u_tokprob` row = effect of
a shift correction alone; bottom-right = both. Pick the MVP configuration (smallest gap at acceptable coverage and cost).